# 3-3: Webscraping

## Web scraping with BeautifulSoup

Web scraping is programmatically collecting information from various websites. While there are many libraries and frameworks in various languages that can extract web data, Python has long been a popular choice because of its plethora of options for web scraping.

## Ethical web scraping
Before choosing to engage in web scraping, you always have to consider some things:
1. Many websites have a Terms of Use which may not allow scraping. We must respect websites that do not want to be scraped.
2. Is there an API available already? If so, there's no need for us to write a scraper. APIs are created to provide access to data in a controlled way as defined by the owners of the data, so we prefer to use APIs if they're available.
3. Making requests to a website can cause a toll on a website's performance. A web scraper that makes too many requests can be as debilitating. We must scrape responsibly so we won't cause any disruption to the regular functioning of the website.

If you have doubts about the ethics of scraping some website, please consult with me.


## Scraping from Wikipedia
We're going to scrape some information from Wikipedia, which has a simple page layout with a consistent template.

For web scraping we're going to need two libraries: [requests](https://requests.readthedocs.io/en/master/) and [BeautifulSoup](https://www.crummy.com/software/BeautifulSoup/bs4/doc/). BeautifulSoup is what we use to actually navigate and parse the page that we're scraping. We'll import the `time` library too. This will allow us to `time.sleep(5)` so that we don't overload anyone's servers. 

We will talk a little about HTML and CSS - you need to know more about these if you want to get good at web scraping. Here's a good point to start: [What are HTML and CSS?](https://html.com/) 

If you're looking for a quick crash course in developer tools for HTML and CSS, check out this [YouTube video](https://www.youtube.com/watch?v=FQKvro1Wz-E).

In [ ]:
# !pip install beautifulsoup4

In [ ]:
import requests
from bs4 import BeautifulSoup
import time
import pandas as pd

<img src="../img/html-image-tag.png" alt="source" style="width: 400px;"/>

### For this exercise, we will scrape all the citations on the Wikipedia "Data Science" page

First we use requests to make a `.get` request to the page. First, hav a look at what's on the [Data science](https://en.wikipedia.org/wiki/Data_science) Wikipedia page. Next, we'll access this page using a GET request through the `requests` library.

In [ ]:
r = requests.get('https://en.wikipedia.org/wiki/Data_science')

We now have an .html object. There is no .html method in the requests library (like for json), but BeautifulSoup will help us get there. First, extract the html string:

In [ ]:
source = r.text
source

Hmmm, are you getting a message that reads something like this? "Please set a user-agent and respect our robot policy [https://w.wiki/4wJS](https://w.wiki/4wJS). See also [https://phabricator.wikimedia.org/T400119](https://phabricator.wikimedia.org/T400119)"

What is this?

It's because `requests` sends a default __user agent__ (the listed user that Wikipedia sees on their end), something like `python-requests/2.x`. Wikimedia’s policy says generic/default user agents may be blocked, and scripts should identify themselves with contact information. That blocked response is what you’re seeing instead of the Wikipedia page.

We need to follow their policies. To do that, we'll add some info to our request using Wikipedia's preferred `headers`, like this:

In [ ]:
url = "https://en.wikipedia.org/wiki/Data_science"

headers = {
    "User-Agent": "DIGHUM101-WebScraping/0.1 (kollmer2@illinois.edu) python-requests",
    "Accept-Encoding": "gzip"
}

r = requests.get(url, headers=headers, timeout=10)
r.raise_for_status()

source = r.text

Did that work? Let's see what happens when we look at `source` now:

In [ ]:
print(source)

Neat! If you visit the Data Science Wikipedia page, right click with your mouse and click "View source" - it's the same thing! 

<img src="../img/page_source.gif" alt="source" style="width: 400px;"/>

Now we convert it into a BeautifulSoup object that makes navigating the HTML tree much easier.

Note that Beautiful Soup offers a number of ways to customize how the parser treats incoming HTML and XML. We are using the `html.parser` parser here, but we could use [different ones](https://www.crummy.com/software/BeautifulSoup/bs4/doc/#differences-between-parsers) as well. It all depends on the website you're trying to scrape.

In [ ]:
soup = BeautifulSoup(source, "html.parser")
print(type(soup))
print(soup)

Then, use the `.prettify()` method to look at the HTML, and even get a slice of it. Let's take a look at what we have:

In [ ]:
print(soup.prettify())

Let's use BeautifulSoup functions to find things on a page, such as:

1. HTML tags
2. HTML Attributes
3. CSS Selectors

Let's search first for **HTML tags**. 

The function `find_all` searches the `soup` tree to find all the elements with a particular HTML tag, and returns a list of all those elements. Let's search for all of the [`a` tags](https://www.w3schools.com/tags/tag_a.asp) (i.e., hyperlinks).

In [ ]:
soup.find_all("a")

Since the `.find_all()` method is used so frequently, there is a shortcut for it. You can just treat the soup object itself as a function, and pass it the tag you're looking for as an argument.

So `soup.find_all('a')` is the same as `soup('a')`:

In [ ]:
soup.find_all('a') == soup('a')

You probably noticed that `.find_all()` returned a lot of elements, most of which we might not want. One way to narrow down our search is to specify that we're only looking for elements that have a certain CSS class. Alternatively we can use the `.select()` method. We pass an argument to the method that consists of the tag and the CSS class separated by a period. For instance, we can grab the title with the following CSS selector:

In [ ]:
soup.select("h1.firstHeading")

How are we getting all these tag and attribute names? Typically, you will want to go to a web page on your browser, right-click on an element you're interested in (such as the heading in the example above) and select "inspect" in order to see the HTML and CSS that makes up the web page. You can then also navigate to other elements in the HTML.

<img src="../img/inspect.gif" alt="inspect" style="width: 400px;"/>

## Scraping text

Inspecting the HTML, we can see there's a tag with an id called `bodyContent`, where all the main text of the article can be found. Let's retrieve it.

In [ ]:
# 'mw-content-text' is an attribute
body = soup.find(id="mw-content-text")
body

In [ ]:
type(body)

Once we identify elements, we want to access the information in a certain element. This usually means two things:

1. Text
2. Attributes

Here, our `body` variable here is a BeautifulSoup `Tag` object. This means it has a `text` attribute. Let's grab all the `p` (paragraph) tags from our resulting BeautifulSoup object and print these `text` attributes.

In [ ]:
for t in body.find_all("p"):
    print(t.text)

## Scraping links 

Next, let's find all the places in the text where there is a link to another website. Using the `.find()` method, we can find all the links on the page that are within the main text. 

Note that we have a special beautifulSoup `Tag` object, meaning we can use its methods on our `text` variable as well. Let's use the `.attrs` attribute to see the attributes for the first `a` tag (i.e., the first hyperlink in this BeautifulSoup object). We can get that with indexing :)

In [ ]:
first_link = body("a")[0].attrs
print(first_link)

You'll notice that it looks a lot like a dictionary, so we can index it as such. Since we want the link, we can use the `href` attribute like a dictionary key to get the corresponding value.

In [ ]:
first_link['href']

In [ ]:
# We can also use .get() to access attributes
first_link.get('href')

# This method is safer as it returns None if the attribute does not exist

Knowing this, we can now iterate over all `a` tags and access them as dictionaries to retrieve the ["href" attribute](https://www.w3schools.com/tags/att_a_href.asp), which specifies the URL of the page the link goes to.

In [ ]:
for line in body.find_all('a'):
    href = line.get('href')  # ← returns None if 'href' doesn't exist
    if href:
        print(href)

## Scraping references
Next, let's get the references one can find at the bottom of a Wikipedia page. Let's `find` the references part of the website first and save that to a new variable.

When we inspect the current Wikipedia HTML, the reference section is wrapped in a `div` whose class includes `mw-references-wrap`. We can use `.find()` when we know the tag name (`div`) and one attribute (`class_="mw-references-wrap"`). We write `class_` because `class` is a Python keyword.

Remember: `.find()` returns the first matching tag. If it cannot find one, it returns `None`, which is why we check before moving on.

In [ ]:
refs = soup.find("div", class_="mw-references-wrap")

if refs is None:
    raise ValueError("Could not find the references section. Inspect the page and update the class name.")

print(type(refs))
print(refs.name)
print(refs.get("class"))

Now that `refs` is a BeautifulSoup `Tag`, we can search inside only the references section instead of searching the whole page. First, let's use `.find_all()`: we give BeautifulSoup the tag name `span`, and then the class attribute `reference-text`. Each `span.reference-text` is one citation entry.

In [ ]:
citation_spans_find = refs.find_all("span", class_="reference-text")

print(len(citation_spans_find))
first_citation = citation_spans_find[0]
first_citation

Next, we'll `select` the first `reference-text` attribute.

`select()` uses CSS selector syntax. The selector `span.reference-text` means: find every `span` tag whose class is `reference-text`. In this case, we could either use `find_all` or `select`; usage often depends on the use case. See [here](https://stackoverflow.com/questions/38028384/beautifulsoup-difference-between-find-and-select) if you want to learn more.

In [ ]:
citation_spans_select = refs.select("span.reference-text")

print(len(citation_spans_select))
first_citation = citation_spans_select[0]
print(first_citation == citation_spans_find[0])
first_citation

That gives us the same kind of BeautifulSoup object as the `find_all()` version. Let's check out its type.

In [ ]:
# check out its type
print(type(first_citation))

If we want to get the link to this citation, we just have to navigate to it. We can again find whatever `a` elements are in this tag, just like we did before. To stay with the `find` family first, we'll use `.find_all("a")` inside the one citation.

In [ ]:
# Find the "a" elements
citation_links_find = first_citation.find_all("a")
print(citation_links_find)

As you can see, this returns a list. We can write the same search with CSS selector syntax too: `first_citation.select("a")` selects all the anchor tags inside `first_citation`.

In [ ]:
citation_links_select = first_citation.select("a")
print(citation_links_select == citation_links_find)

Note that we have a special BeautifulSoup `Tag` object. Let's use indexing to get the first `a` tag, then use the `.attrs` attribute to see its attributes.

In [ ]:
# Get the first one
first_link = citation_links_find[0]
print(first_link)
print(first_link.attrs)

Since we want the link, we can use the `href` attribute again to get the corresponding value. We'll use `.get('href')` because it returns `None` instead of an error if an `a` tag does not have an `href`.

In [ ]:
first_link.get("href")

Now, get all the links contained in the references and add them to a list. We'll combine the two ideas: use `select()` to get every citation (`span.reference-text`), then use `find_all()` inside each citation to get the `a` tags. Finally, we keep only links that are not internal Wikipedia article links.

In [ ]:
# make accumulator list
refs_list = []

# start at the endnotes
references = refs.select("span.reference-text")

# loop through references
for ref in references:
    # each citation can contain several links
    for a_element in ref.find_all("a"):
        link = a_element.get("href")
        
        # get rid of links to wiki articles and citation backlinks
        if link and not link.startswith("/wiki") and not link.startswith("#"):
            refs_list.append(link)

refs_list

In [ ]:
# Convert to data frame
citations_df = pd.DataFrame(refs_list, columns = ["Citation"])
citations_df.head()

In [ ]:
# Export to .csv
citations_df.to_csv("citations.csv")

## Iterating Over Webpages

For a second example, let's scrape product pages from [Evening Land Books](https://eveninglandbooks.com). This site is built with WordPress and WooCommerce, which means product pages follow a fairly consistent HTML template. That template is what makes this scrape possible: the product name is stored in the product title area, and the short description is stored in the WooCommerce short-description area.

I own this website and business, and I write the product short descriptions myself, so we have permission to use it for this classroom example. But we still want to scrape carefully: we'll identify our script with a user-agent, set a `timeout` so requests do not hang forever, and pause between requests so we don't overwhelm the site.

Why look at this example? Because it gives us an opportunity to consider how to iterate over pages. Quite often, webscraping requires iterating over many pages. Maybe we've identified a pattern across a site and we want to extract a bunch of data across numerous pages. Maybe we have a list of URLs we need to scrape over. In cases like these, we need to design our webscraping process to respect rate limits, use `timeout` and pauses, and iterate carefully. To identify patterns in urls, we need to investigate sites by hand––look over urls carefully, check the developer view.

Since I'm the site developer at Evening Land Books, I already know product URLs follow this pattern:

`https://eveninglandbooks.com/product/{SKU_number}/`

The SKU numbers are seven digits long. For example, product 1 is written as `0000001`, not just `1`.

With that in mind, we'll build a scraper in small pieces:

1. Store the repeated settings: the base URL, the headers, and the delay.
2. Write one helper function to turn a number into a seven-digit SKU.
3. Write another helper function to turn that SKU into a product URL.
4. Use those pieces inside a loop.

First, let's save the values that we'll reuse.

`BASE_URL` is the part of every product URL that stays the same. `HEADERS` tells the website who is making the request. `DELAY_SECONDS` controls how long we pause between requests. Changing that one number later will change the delay everywhere we use it.

In [ ]:
BASE_URL = "https://eveninglandbooks.com/product"

HEADERS = {
    "User-Agent": "DIGHUM101-WebScraping/0.1 (kollmer2@illinois.edu)",
    "Accept-Encoding": "gzip, deflate"
}

DELAY_SECONDS = 1.0

Next, let's write a helper function for SKUs. A helper function is a small function that does one repeated job for us. Here, the job is to turn an integer like `1` into a seven-digit string like `0000001`.

The expression `f"{sku_number:07d}"` is an f-string. The `sku_number` part says which variable to format. The `07d` part says: format it as an integer with seven digits, and use zeroes to fill any empty places on the left.

In [ ]:
def make_sku(sku_number):
    sku = f"{sku_number:07d}"
    return sku

print(make_sku(1))
print(make_sku(25))
print(make_sku(1600))

Now we can use `make_sku()` inside another helper function. This second function builds the full product URL.

Notice the order of operations: first we make the SKU, then we place that SKU into the URL. This is a common pattern in programming: one small helper function can be used inside another function.

In [ ]:
def make_product_url(sku_number):
    sku = make_sku(sku_number)
    url = f"{BASE_URL}/{sku}/"
    return sku, url

make_product_url(1)

Before we try many pages, we should always inspect one page. This lets us confirm that the URL works and gives us HTML that BeautifulSoup can parse.

The `timeout=10` argument tells `requests` to give up after 10 seconds instead of waiting forever. The status code tells us whether the request succeeded. A status code of `200` means the page loaded successfully.

In [ ]:
sku, url = make_product_url(1)

elb_response = requests.get(url, headers=HEADERS, timeout=10)

print(sku)
print(url)
print(elb_response.status_code)

elb_source = elb_response.text
elb_source[:500]

Now we parse the HTML. Because this is a WooCommerce product page, we can use WooCommerce's classes as CSS selectors:

- `h1.product_title` selects the product title.
- `div.woocommerce-product-details__short-description` selects the short description.
- `span.sku` selects the SKU shown in the product metadata.

For more info about WooCommerce classes, check out the [documentation here.](https://developer.woocommerce.com/docs/extensions/core-concepts/class-reference/)

The method `.select_one()` returns the first matching element. Then `.get_text(" ", strip=True)` extracts readable text, uses spaces between pieces of text, and removes extra whitespace at the beginning and end.

In [ ]:
elb_soup = BeautifulSoup(elb_source, "html.parser")

title_tag = elb_soup.select_one("h1.product_title")
short_description_tag = elb_soup.select_one("div.woocommerce-product-details__short-description")
sku_tag = elb_soup.select_one("span.sku")

sku_on_page = sku_tag.get_text(strip=True)
product_name = title_tag.get_text(" ", strip=True)
short_description = short_description_tag.get_text(" ", strip=True)

print(sku_on_page)
print(product_name)
print(short_description)

Once we know the selectors work on one page, we can turn that logic into a function. This function does one job: given a SKU number, it tries to scrape that product page.

To keep this beginner example readable, we will handle only one common problem: the page does not load successfully. If the status code is not `200`, the function returns `None`. If the page does load, the function returns a dictionary with four values: SKU, URL, product name, and short description.

In [ ]:
def scrape_elb_product(sku_number):
    sku, url = make_product_url(sku_number)
    response = requests.get(url, headers=HEADERS, timeout=10)
    
    if response.status_code != 200:
        return None
    
    soup = BeautifulSoup(response.text, "html.parser")
    title_tag = soup.select_one("h1.product_title")
    short_description_tag = soup.select_one("div.woocommerce-product-details__short-description")
    
    product_name = title_tag.get_text(" ", strip=True)
    short_description = short_description_tag.get_text(" ", strip=True)
    
    return {
        "sku": sku,
        "url": url,
        "product_name": product_name,
        "short_description": short_description
    }

scrape_elb_product(1)

Before running a larger scrape, test the function on a few pages. This is both an ethical scraping practice and a debugging practice: if our selector is wrong, it is much better to discover that after three requests than after hundreds.

The loop below uses `range(1, 4)`, which gives us `1`, `2`, and `3`. Each time through the loop, we scrape one product page, add the result to `test_rows`, print a short progress message, and then pause.

In [ ]:
test_rows = []

for sku_number in range(1, 4):
    product = scrape_elb_product(sku_number)
    
    if product is not None:
        test_rows.append(product)
        print("Collected", product["sku"])
    else:
        print("Skipped", make_sku(sku_number))
    
    time.sleep(DELAY_SECONDS)

pd.DataFrame(test_rows)

Now we can scale up from a tiny test to a larger SKU range. The range `range(START_SKU, END_SKU + 1)` includes both endpoints; without `+ 1`, Python would stop before `END_SKU`.

With a one-second delay, checking thousands of possible SKU numbers would take thousands of seconds. You should always keep that in mind while webscraping. Sometimes it is best to let a scraping script run overnight. I've scraped for several days at a time!

Also pay attention to how `END_SKU` is set to a smaller number first. You could set this to a larger number, but please do not do that during class. Keep your scraping to a minimum to respect websites, servers, and developers. If you wanted to scrape the whole list of products, it would also be best to:

1. Reach out to the site owner and see if you can get the data directly.
2. Check the site's webscraping policies online.
3. Make sure you know what pages, and how many pages, you plan to iterate over. In this example, you should first investigate how many SKUs the site has.

In [ ]:
START_SKU = 1000
END_SKU = 1020

product_rows = []

for sku_number in range(START_SKU, END_SKU + 1):
    product = scrape_elb_product(sku_number)
    
    if product is not None:
        product_rows.append(product)
    
    print("Checked", make_sku(sku_number), "| collected", len(product_rows), "products")
    time.sleep(DELAY_SECONDS)

elb_products_df = pd.DataFrame(product_rows)
elb_products_df.head()

Finally, save the rows as a CSV. We use `index=False` so pandas does not add an extra unnamed index column to the file. The resulting file will contain the SKU, URL, product name, and short description.

In [ ]:
elb_products_df.to_csv("elb_products.csv", index=False)

print(f"Saved {len(elb_products_df)} products to elb_products.csv")